<a href="https://colab.research.google.com/github/budennovsk/Pandas/blob/master/v4_cold_start.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install implicit catboost rectools[lightfm] replay-rec==0.21.2rc0 faiss-cpu #replay-rec !pip install faiss-cpu
# !pip -q uninstall -y pyspark
# !pip -q install "pyspark==3.4.3"


In [ ]:
from pathlib import Path
import typing as tp
import warnings

import pandas as pd
import numpy as np

from implicit.nearest_neighbours import CosineRecommender
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import RidgeClassifier
from catboost import CatBoostClassifier, CatBoostRanker
try:
    from lightgbm import LGBMClassifier, LGBMRanker
    LGBM_AVAILABLE = True
except ImportError:
    warnings.warn("lightgbm is not installed. Some parts of the notebook will be skipped.")
    LGBM_AVAILABLE = False
from rectools.dataset import Interactions
from lightfm import LightFM
from rectools import Columns
from rectools.dataset import Dataset
from rectools.metrics import Precision, Recall, MeanInvUserFreq, Serendipity,MAP,NDCG,HitRate,calc_metrics
from rectools.models import (
    ImplicitALSWrapperModel,
    ImplicitBPRWrapperModel,
    LightFMWrapperModel,
    PureSVDModel,
    ImplicitItemKNNWrapperModel,
    EASEModel,
    RandomModel,
    PopularInCategoryModel,
    PopularModel)
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking
from implicit.nearest_neighbours import CosineRecommender
from rectools.models.base import ExternalIds
from rectools.models.ranking import (
    CandidateRankingModel,
    CandidateGenerator,
    Reranker,

    CatBoostReranker,
    CandidateFeatureCollector,
    PerUserNegativeSampler
)
from rectools.model_selection import cross_validate, TimeRangeSplitter,LastNSplitter,Splitter
from tqdm.auto import tqdm

In [ ]:
path_users = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/users.csv'
path_items = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/items.csv'
path_interactions = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/interactions.csv'


users = pd.read_csv(path_users)
items = pd.read_csv(path_items)
interactions = (
    pd.read_csv(path_interactions, parse_dates=["last_watch_dt"])
    .rename(columns={"last_watch_dt": Columns.Datetime})
)
interactions = interactions.head(1000000)
users_clise = users[users['user_id'].isin(interactions['user_id'].unique())]
items_clise = items[items['item_id'].isin(interactions['item_id'].unique())]
interactions["weight"] = 1
dataset = Dataset.construct(interactions)
RANDOM_STATE = 32


# dataset
# interactions[Columns.Datetime] = pd.to_datetime(interactions[Columns.Datetime], format='%Y-%m-%d')

In [ ]:
max_date = interactions[Columns.Datetime].max()
train = interactions[interactions[Columns.Datetime] < max_date - pd.Timedelta(days=7)].copy()
test = interactions[interactions[Columns.Datetime] >= max_date - pd.Timedelta(days=7)].copy()

print(f"train: {train.shape}")
print(f"test: {test.shape}")

In [ ]:
# отфильтруем холодных пользователей из теста
cold_users = list(set(test['user_id']) - set(train['user_id']))
len(cold_users)

In [ ]:
test['user_id'].nunique(), len(cold_users) +test[test['user_id'].isin(train['user_id'].unique())]['user_id'].nunique()

In [ ]:
test['datetime'].min(), test['datetime'].max()

In [ ]:
train['datetime'].min(), train['datetime'].max()

In [ ]:
train[train['user_id'].isin(cold_users)]['user_id'].nunique()

In [ ]:
test[test['user_id'].isin(train['user_id'].unique())]['user_id'].nunique()

In [ ]:
train[train['user_id']==626036]

In [ ]:
test[test['user_id']==626036]

In [ ]:
test_cold=test[test['user_id'].isin(cold_users)]
test_cold[test_cold['user_id'].isin(cold_users)]['user_id'].nunique()

In [ ]:
valid_user_ids = set(users_clise["user_id"].dropna().unique())
valid_item_ids = set(items_clise["item_id"].dropna().unique())

train_in_all = train[
    train["user_id"].isin(valid_user_ids) &
    train["item_id"].isin(valid_item_ids)
].copy()


users_clise_train = users_clise[users_clise['user_id'].isin(train_in_all['user_id'].unique())]
# users_clise_train['user_id'].nunique()

items_clise_train = items_clise[items_clise['item_id'].isin(train_in_all['item_id'].unique())]
# items_clise_train['item_id'].nunique()

In [ ]:
train_in_all["user_id"].nunique(), users_clise_train["user_id"].nunique()

In [ ]:
# train_dataset_all = Dataset.construct(interactions_df=train_in_all)

In [ ]:
items_clise_train["genre"] = items_clise_train["genres"].str.split(",")
genre_feature = items_clise_train[["item_id", "genre"]].explode("genre")
genre_feature.columns = ["id", "value"]
genre_feature["feature"] = "genre"
genre_feature_train = genre_feature[genre_feature['id'].isin(train['item_id'])]
genre_feature_train.head()

In [ ]:
import re

col = "value"

def canon(x: str) -> str:
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = x.replace("–", "-").replace("—", "-")
    x = re.sub(r"\s*-\s*", "-", x)  # "ток - шоу" -> "ток-шоу"
    return x

# 1) нормализуем текст
genre_feature_train[col] = genre_feature_train[col].map(canon)

# 2) склейки (синонимы/варианты)
# mapping = {
#     "советские": "русские",
#     # (по написанию)
#     "токшоу": "ток-шоу",
#     "реалити": "реалити-шоу",
#     "шоу": "телешоу",  # если хочешь объединять "шоу" с "телешоу"
#     "мультфильм": "мультфильмы",
#     "русские мультфильмы": "мультфильмы",  # если хочешь все мультфильмы в одну категорию
#     "западные мультфильмы": "мультфильмы",
#     "детские": "для детей",
#     "для самых маленьких": "для детей",    # если хочешь объединять
# }
mapping = {
    "советские": "русские",
    # (по написанию)
    "токшоу": "ток-шоу",
    "реалити": "реалити-шоу",
    "шоу": "телешоу",  # если хочешь объединять "шоу" с "телешоу"
    "мультфильм": "мультфильмы",
    "русские мультфильмы": "мультфильмы",  # если хочешь все мультфильмы в одну категорию
    "западные мультфильмы": "мультфильмы",
    "детские": "для детей",
    "для самых маленьких": "для детей",
    "исторические":	"историческое",
    "музыка":	"музыкальные",
    "музыка":	"мюзиклы",
    "комиксы":"по комиксам",
    "спорт":"футбол",
    "комедии":	"юмор",
}

genre_feature_train[col] = genre_feature_train[col].replace(mapping)
genre_feature_train[col].nunique()

In [ ]:

counts = (
    genre_feature_train
    .groupby('value', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

counts_lt10 = counts[counts["count"] < 10].reset_index(drop=True)

counts_lt10  # все value, которые встречаются меньше 10 раз

In [ ]:
rare_values = set(counts_lt10["value"])

genre_feature_train["value"] = genre_feature_train["value"].where(
    ~genre_feature_train["value"].isin(rare_values),
    "other",
)
genre_feature_train[col].nunique()

In [ ]:
# import re
# import numpy as np
# import pandas as pd
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity

# df = genre_feature_train
# col = "value"

# def canon(x: str) -> str:
#     x = str(x).strip().lower()
#     x = re.sub(r"\s+", " ", x)
#     x = x.replace("–", "-").replace("—", "-")
#     x = re.sub(r"\s*-\s*", "-", x)
#     return x

# # уникальные жанры
# genres = sorted(pd.Series(df[col]).map(canon).dropna().unique().tolist())

# model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# emb = model.encode(genres, normalize_embeddings=True)

# sim = cosine_similarity(emb)

# threshold = 0.72  # подбирается: 0.68–0.80 (выше = строже)
# pairs = []
# n = len(genres)
# for i in range(n):
#     for j in range(i+1, n):
#         s = float(sim[i, j])
#         if s >= threshold:
#             pairs.append((genres[i], genres[j], s))

# pairs = sorted(pairs, key=lambda x: -x[2])
# pairs_df = pd.DataFrame(pairs, columns=["a", "b", "sim"])

# # посмотри топ совпадений
# pairs_df.head(50)

In [ ]:
# dataset_feature_train_all = Dataset.construct(
#     interactions_df=train_in_all,
#     user_features_df=None,
#     item_features_df=genre_feature_train,
#     cat_item_features=['genre']
# )

In [ ]:
def plot_normalized_barh_chart(
    dataframe: pd.DataFrame,
    column_name: str,
    title: str,
    figsize: tp.Tuple[int, int] = (12, 7),
    annotate_chart: bool = True,
    xlabel: tp.Optional[str] = None,
    ylabel: tp.Optional[str] = None,
):
    ax = (
        dataframe[column_name]
        .value_counts(dropna=False, normalize=True)
        .sort_index()
        .plot(
            kind='barh',
            grid=True,
            title=title,
            figsize=figsize,
            xlabel=xlabel,
            ylabel=ylabel,
        )
    )

    if annotate_chart:
        for bars in ax.containers:
            ax.bar_label(bars, labels=[f'{x:.1%}' for x in bars.datavalues])

In [ ]:
users_clise_train["sex"] = users_clise_train["sex"].map({"Ж": 1, "М": 0}).fillna(-1).astype("int8")
age_category = pd.CategoricalDtype(
    categories=[
         "unknown",
        'age_18_24',
        'age_25_34',
        'age_35_44',
        'age_45_54',
        'age_55_64',
        'age_65_inf',
    ],
    ordered=True,
)
users_clise_train["age"] = (
    users_clise_train["age"]
    .fillna("unknown")
    .astype(age_category)
)
income_category = pd.CategoricalDtype(
    categories=[
        "unknown",          # добавили
        "income_0_20",
        "income_20_40",
        "income_40_60",
        "income_60_90",
        "income_90_150",
        "income_150_inf",
    ],
    ordered=True,
)

# если income уже строки/категории — заполним nan и приведем тип
users_clise_train["income"] = (
    users_clise_train["income"]
    .fillna("unknown")
    .astype(income_category)
)
YEAR_FROM = 1990
STEP_SIZE = 5
bins = [year for year in range(YEAR_FROM, int(items_clise['release_year'].max()) + STEP_SIZE, STEP_SIZE)]
bins = [int(items_clise['release_year'].min())] + bins
items_clise['year_bin'] = pd.cut(items_clise['release_year'],
                           bins=bins, include_lowest=True)

In [ ]:
users_clise_train['age'].value_counts()

In [ ]:
plot_normalized_barh_chart(users_clise_train, 'sex', 'Распределение выхода контента')

In [ ]:
user_features_frames = []
for feature in ["sex", "age", "income"]:
    feature_frame = users_clise_train.reindex(columns=[Columns.User, feature])
    feature_frame.columns = ["id", "value"]
    feature_frame["feature"] = feature
    user_features_frames.append(feature_frame)
user_features_train = pd.concat(user_features_frames)
user_features_train.tail()

In [ ]:
content_feature_train = items_clise_train.reindex(columns=[Columns.Item, "content_type"])
content_feature_train.columns = ["id", "value"]
content_feature_train["feature"] = "content_type"

In [ ]:
item_features_train = pd.concat((genre_feature_train,content_feature_train))
item_features_train

In [ ]:
item_features_train,user_features_train

In [ ]:
dataset_train_i_u_features = Dataset.construct(
    interactions_df=train_in_all,
    user_features_df=user_features_train,
    cat_user_features=["sex", "age", "income"],
    item_features_df=item_features_train,
    cat_item_features=["genre", "content_type"],
)

In [ ]:
# Take few models to compare
from datetime import timedelta

models = {
    "popular": PopularModel(popularity="n_interactions"),
    "popular_7d": PopularModel(period=timedelta(days=7)),
    "random": RandomModel(random_state=RANDOM_STATE),
    # "most_raited": PopularModel(popularity="mean_weight"),
    "pop_cat": PopularInCategoryModel(category_feature='genre', n_categories=20),
    # "cosine_knn": ImplicitItemKNNWrapperModel(CosineRecommender()),
    # 'iALS':ImplicitALSWrapperModel(
    #       AlternatingLeastSquares(
    #         factors=10,  # latent embeddings size
    #         regularization=0.1,
    #         iterations=10,
    #         alpha=50,  # confidence multiplier for non-zero entries in interactions
    #         random_state=RANDOM_STATE)),
    'LightFM':LightFMWrapperModel(
            LightFM(no_components=32,
                    loss="warp",
                    random_state=RANDOM_STATE)),

}

# We will calculate several classic (precision@k and recall@k) and "beyond accuracy" metrics
metrics = {
    "prec@10": Precision(k=10),
    "recall@10": Recall(k=10),
    "novelty@10": MeanInvUserFreq(k=10),
    # "serendipity@10": Serendipity(k=10),
    "MAP@10": MAP(k=10),
    "NDCG@10": NDCG(k=10),
    "HitRate@10": HitRate(k=10)
}

K_RECS = 10

In [ ]:
results = []
K_RECOS = 10
RANDOM_STATE = 42
NUM_THREADS =-1
for model_name, model in tqdm(models.items()):
    model_quality = {'model': model_name}

    model.fit(dataset_train_i_u_features)
    recos = model.recommend(
        users=test_cold[Columns.User].unique(),
        dataset=dataset_train_i_u_features, #dataset_feature_train train_dataset
        k=K_RECOS,
        filter_viewed=True,
    )

    metric_values = calc_metrics(metrics, recos, test_cold, train)

    model_quality.update(metric_values)
    results.append(model_quality)

In [ ]:
test_cold[Columns.User].nunique(),

In [ ]:
train_in_all[train_in_all['user_id'].isin(test_cold[Columns.User].unique())]

In [ ]:
user_features_train[user_features_train['id'].isin(test_cold[Columns.User].unique())], item_features_train[item_features_train['id'].isin(test[test['user_id'].isin(test_cold[Columns.User].unique())]['item_id'].unique())]

In [ ]:
test[test['user_id'].isin(test_cold[Columns.User].unique())]['item_id'].nunique()

In [ ]:
df_quality = pd.DataFrame(results).T

df_quality.columns = df_quality.iloc[0]

df_quality.drop('model', inplace=True)
# df_quality.style.highlight_max(color='lightgreen', axis=1)
df_quality

In [ ]:
n_splits = 3

splitter = TimeRangeSplitter(
    test_size="7D",
    n_splits=n_splits,
    filter_already_seen=True,
    filter_cold_items=False,
    filter_cold_users=False,
)
cv_results = cross_validate(
    dataset=dataset_train_i_u_features,
    splitter=splitter,
    models=models,
    metrics=metrics,
    k=K_RECS,
    filter_viewed=True,
)

In [ ]:
pivot_results = (
    pd.DataFrame(cv_results["metrics"])
    .drop(columns="i_split")
    .groupby(["model"], sort=False)
    .agg(["mean"])
)
pivot_results

In [ ]:
# pip install scikit-learn faiss-cpu pandas numpy

import numpy as np
import pandas as pd
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# =====================
# REQUIRED INPUTS (must exist in your notebook/session):
#   users_clise: DataFrame with user features (user_id, age, income, sex, kids_flg)
#   train_in_all: interactions for TRAIN users (must include user_id, item_id)
#   test_cold: interactions/gt for TEST cold users (must include user_id, item_id)
# =====================

U_USER_ID, U_AGE, U_INCOME, U_SEX, U_KIDS = "user_id", "age", "income", "sex", "kids_flg"
I_ITEM_ID = "item_id"

K_NEIGHBORS = 10
TOPK_ITEMS = 10
SVD_DIM = 32
RANDOM_STATE = 42

# ---------- helpers ----------
def ensure_cols(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = ""
        df[c] = df[c].fillna("").astype(str)
    return df

def build_user_docs(users: pd.DataFrame) -> list[str]:
    return (
        "sex " + users[U_SEX] + " "
        "age " + users[U_AGE] + " "
        "income " + users[U_INCOME] + " "
        "kids " + users[U_KIDS]
    ).tolist()

def dcg_at_k(hits, k):
    dcg = 0.0
    for i, h in enumerate(hits[:k], start=1):
        if h:
            dcg += 1.0 / np.log2(i + 1)
    return dcg

def ndcg_at_k(rec_list, truth_set, k=10):
    if truth_set is None or len(truth_set) == 0:
        return 0.0
    hits = [1 if r in truth_set else 0 for r in rec_list[:k]]
    dcg = dcg_at_k(hits, k)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, min(len(truth_set), k) + 1))
    return float(dcg / idcg) if idcg > 0 else 0.0

def ap_at_k(rec_list, truth_set, k=10):
    if truth_set is None or len(truth_set) == 0:
        return 0.0
    hits = [1 if r in truth_set else 0 for r in rec_list[:k]]
    denom = min(len(truth_set), k)
    if denom == 0:
        return 0.0
    ap = 0.0
    cum = 0
    for i, h in enumerate(hits, start=1):
        if h:
            cum += 1
            ap += cum / i
    return float(ap / denom)

def precision_recall_hitrate_at_k(rec_list, truth_set, k=10):
    if truth_set is None or len(truth_set) == 0:
        return 0.0, 0.0, 0.0
    rec_k = rec_list[:k]
    hit_count = sum(1 for r in rec_k if r in truth_set)
    prec = hit_count / k
    rec = hit_count / len(truth_set)
    hitrate = 1.0 if hit_count > 0 else 0.0
    return prec, rec, hitrate

def build_truth_dict(df_gt: pd.DataFrame) -> dict:
    df_gt = df_gt[[U_USER_ID, I_ITEM_ID]].copy()
    df_gt[U_USER_ID] = df_gt[U_USER_ID].astype(str)
    df_gt[I_ITEM_ID] = df_gt[I_ITEM_ID].astype(str)
    return df_gt.groupby(U_USER_ID)[I_ITEM_ID].apply(set).to_dict()

# =====================
# 1) Split users
# =====================
users_train = users_clise[~users_clise[U_USER_ID].isin(test_cold[U_USER_ID])].copy()
users_test  = users_clise[ users_clise[U_USER_ID].isin(test_cold[U_USER_ID])].copy()

users_train = ensure_cols(users_train, [U_SEX, U_AGE, U_INCOME, U_KIDS])
users_test  = ensure_cols(users_test,  [U_SEX, U_AGE, U_INCOME, U_KIDS])

# =====================
# 2) Fit user-embeddings model on TRAIN USERS (TF-IDF + SVD)
# =====================
tfidf_u = TfidfVectorizer(
    lowercase=True,
    min_df=2,
    max_features=500,
    ngram_range=(1, 2),
)
X_u_train = tfidf_u.fit_transform(build_user_docs(users_train))

svd_u = TruncatedSVD(n_components=SVD_DIM, random_state=RANDOM_STATE)
emb_u_train = svd_u.fit_transform(X_u_train).astype("float32")
faiss.normalize_L2(emb_u_train)

# =====================
# 3) FAISS index over train users
# =====================
index_u = faiss.IndexFlatIP(emb_u_train.shape[1])
index_u.add(emb_u_train)

train_user_ids = users_train[U_USER_ID].astype(str).values

# =====================
# 4) Embed TEST users in same space
# =====================
X_u_test = tfidf_u.transform(build_user_docs(users_test))
emb_u_test = svd_u.transform(X_u_test).astype("float32")
faiss.normalize_L2(emb_u_test)

# =====================
# 5) Nearest neighbors (user-user)
# =====================
sims, nn_idx = index_u.search(emb_u_test, K_NEIGHBORS)
nn_user_ids = train_user_ids[nn_idx]  # shape: [n_test, K_NEIGHBORS]

# =====================
# 6) Precompute train user -> items list from train interactions
# =====================
train_in_all2 = train_in_all[[U_USER_ID, I_ITEM_ID]].copy()
train_in_all2[U_USER_ID] = train_in_all2[U_USER_ID].astype(str)
train_in_all2[I_ITEM_ID] = train_in_all2[I_ITEM_ID].astype(str)

user2items = train_in_all2.groupby(U_USER_ID)[I_ITEM_ID].apply(list).to_dict()

# =====================
# 7) Build recommendations by aggregating neighbors' items
# =====================
test_user_ids = users_test[U_USER_ID].astype(str).values

recs = []
for i, uid in enumerate(test_user_ids):
    cand_scores = {}

    for j in range(K_NEIGHBORS):
        nb = nn_user_ids[i, j]
        w = float(sims[i, j])           # cosine similarity
        # optional: additional rank decay
        # w = w / (j + 1)

        for it in user2items.get(nb, []):
            cand_scores[it] = cand_scores.get(it, 0.0) + w

    top_items = [it for it, _ in sorted(cand_scores.items(), key=lambda x: x[1], reverse=True)[:TOPK_ITEMS]]
    recs.append(top_items)

out = pd.DataFrame({U_USER_ID: test_user_ids, "recommended_items": recs})
print(out.head(10))

# =====================
# 8) Metrics on test_cold
# =====================
truth = build_truth_dict(test_cold)

prec_list, rec_list, hit_list = [], [], []
ndcg_list, map_list = [], []

for uid, rec_list_u in zip(out[U_USER_ID].values, out["recommended_items"].values):
    gt = truth.get(uid, set())

    prec, rec, hitrate = precision_recall_hitrate_at_k(rec_list_u, gt, k=TOPK_ITEMS)
    ndcg = ndcg_at_k(rec_list_u, gt, k=TOPK_ITEMS)
    ap = ap_at_k(rec_list_u, gt, k=TOPK_ITEMS)

    prec_list.append(prec)
    rec_list.append(rec)
    hit_list.append(hitrate)
    ndcg_list.append(ndcg)
    map_list.append(ap)

metrics = {
    "Precision@10": float(np.mean(prec_list)) if prec_list else 0.0,
    "Recall@10": float(np.mean(rec_list)) if rec_list else 0.0,
    "HitRate@10": float(np.mean(hit_list)) if hit_list else 0.0,
    "NDCG@10": float(np.mean(ndcg_list)) if ndcg_list else 0.0,
    "MAP@10": float(np.mean(map_list)) if map_list else 0.0,
}
print(metrics)

In [ ]:
# твой pivot_results (как есть)            user‑to‑user (kNN) cold-start:
pivot_results = (
    pd.DataFrame(cv_results["metrics"])
    .drop(columns="i_split")
    .groupby(["model"], sort=False)
    .agg(["mean"])
)

# metrics из user-to-user (у тебя уже посчитан dict metrics)
# Приведём названия метрик к тем, что в pivot_results (скорее всего lower-case и @10)
u2u_row = {
    "prec@10": metrics["Precision@10"],
    "recall@10": metrics["Recall@10"],
    "HitRate@10": metrics["HitRate@10"],   # если в pivot_results это поле называется иначе — см. ниже
    "NDCG@10": metrics["NDCG@10"],
    "MAP@10": metrics["MAP@10"],
}

# pivot_results имеет MultiIndex по колонкам (metric, 'mean'), поэтому создаём такой же формат
u2u_df = pd.DataFrame(
    { (k, "mean"): [v] for k, v in u2u_row.items() },
    index=pd.Index(["user_to_user"], name="model")
)

# добавляем строку и выравниваем колонки
pivot_results = pd.concat([pivot_results, u2u_df], axis=0).sort_index()

pivot_results

In [ ]:
# Предполагается, что у вас уже есть pandas DataFrame:
# interactions  : минимум колонки ['user_id', 'item_id'] (может быть больше)
# users_clise   : минимум ['user_id', 'sex', 'age', 'income']
# items_clise   : минимум ['item_id', 'genres', 'content_type']
#
# pip install recbole

import os
import numpy as np
import pandas as pd

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed, get_model, get_trainer

# ----------------------------
# 0) Параметры
# ----------------------------
WORK_DIR = "./recbole_data"
DATASET_NAME = "kion_FM"

SEED = 42
K_RECOS = 10
N_COLD_FRAC = 0.05      # доля cold пользователей (для демонстрации)
N_CANDIDATES = 3000     # кандидаты = top-N популярных items из train

os.makedirs(WORK_DIR, exist_ok=True)
ds_dir = os.path.join(WORK_DIR, DATASET_NAME)
os.makedirs(ds_dir, exist_ok=True)

rng = np.random.default_rng(SEED)

# ----------------------------
# 1) Нормализуем типы и оставляем нужные колонки
# ----------------------------
inter = interactions[["user_id", "item_id"]].dropna().copy()
inter["user_id"] = inter["user_id"].astype(str)
inter["item_id"] = inter["item_id"].astype(str)

users_df = users_clise[["user_id", "sex", "age", "income"]].copy()
users_df["user_id"] = users_df["user_id"].astype(str)


for c in ["sex", "age", "income"]:
    users_df[c] = users_df[c].fillna("unknown").astype(str)

items_df = items_clise[["item_id", "genres", "content_type"]].copy()
items_df["item_id"] = items_df["item_id"].astype(str)
for c in ["genres", "content_type"]:
    items_df[c] = items_df[c].fillna("unknown").astype(str)

# ----------------------------
# 2) Split: cold users (0 интеракций в train)
# ----------------------------
all_users = inter["user_id"].unique()
cold_users = rng.choice(all_users, size=max(1, int(len(all_users) * N_COLD_FRAC)), replace=False)
cold_users = set(cold_users)

inter_test = inter[inter["user_id"].isin(cold_users)].copy()
inter_train = inter[~inter["user_id"].isin(cold_users)].copy()

assert set(inter_train["user_id"].unique()).isdisjoint(set(inter_test["user_id"].unique()))

# ----------------------------
# 3) Экспорт в формат RecBole (.inter/.user/.item)
# ----------------------------
# inter: только TRAIN взаимодействия
inter_out = inter_train.copy()
inter_out.columns = ["user_id:token", "item_id:token"]
# inter_out.to_csv(os.path.join(ds_dir, f"{DATASET_NAME}.inter"), sep="\t", index=False)

# user: все пользователи (и warm, и cold) + их признаки
user_out = users_df.copy()
user_out.columns = ["user_id:token", "sex:token", "age:token", "income:token"]
# user_out.to_csv(os.path.join(ds_dir, f"{DATASET_NAME}.user"), sep="\t", index=False)

# item: все items + их признаки
item_out = items_df.copy()
item_out.columns = ["item_id:token", "genres:token", "content_type:token"]
# item_out.to_csv(os.path.join(ds_dir, f"{DATASET_NAME}.item"), sep="\t", index=False)

# ----------------------------
# 4) Обучаем RecBole LR
# ----------------------------
config = Config(
    model='FM',
    dataset=DATASET_NAME,
    config_dict={
        # "data_path": WORK_DIR,

        "USER_ID_FIELD": "user_id",
        "ITEM_ID_FIELD": "item_id",
        # "LABEL_FIELD": "user_id",  # ← Добавить это

        "load_col": {
            "inter": ["user_id", "item_id", "user_id"],  # ← Добавить user_id
            "user": ["user_id", "sex", "age", "income"],
            "item": ["item_id", "genre", "content_type"],
        },

        "eval_args": {
            "split": {"RS": [1.0, 0.0, 0.0]},
            "mode": "full"  # ← Изменить на labeled для classification
        },

        "metrics": ["Recall", "MRR", "NDCG", "Hit", "Precision", "MAP"],  # ← Метрики для classification
        # topk не нужен

        "epochs": 10,
        "train_batch_size": 4096,
        "learning_rate": 5e-2,

        "seed": SEED,
        "reproducibility": True,
    }
)

init_seed(SEED, True)

dataset = create_dataset(config)
train_data, _, _ = data_preparation(config, dataset)

model = get_model(config["model"])(config, train_data.dataset).to(config["device"])
trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)

# fit без valid_data (метрики не будут вычисляться)
trainer.fit(train_data, valid_data=None, saved=False, show_progress=True)


# ----------------------------
# 5) Рекомендации для cold users (кандидаты = популярные items из TRAIN)
# ----------------------------
top_items = inter_train["item_id"].value_counts().head(N_CANDIDATES).index.to_numpy()

uid_field = dataset.uid_field
iid_field = dataset.iid_field
token2id = dataset.token2id
id2token_item = dataset.field2id_token[iid_field]

cold_user_list = sorted(list(cold_users))

# Важно: cold users должны быть в .user (мы так и сделали)
cold_u_internal = np.array([token2id[uid_field][u] for u in cold_user_list], dtype=np.int64)
cand_i_internal = np.array([token2id[iid_field][i] for i in top_items], dtype=np.int64)

import torch
from recbole.data.interaction import Interaction

device = config["device"]
rows = []

model.eval()
with torch.no_grad():
    for u_ext, u_int in zip(cold_user_list, cold_u_internal):
        u_vec = np.full((len(cand_i_internal),), u_int, dtype=np.int64)
        i_vec = cand_i_internal

        inter_batch = Interaction({
            uid_field: torch.from_numpy(u_vec).to(device),
            iid_field: torch.from_numpy(i_vec).to(device),
        })

        scores = model.predict(inter_batch).detach().cpu().numpy()

        topk_idx = np.argpartition(-scores, kth=K_RECOS - 1)[:K_RECOS]
        topk_sorted = topk_idx[np.argsort(-scores[topk_idx])]

        for rank, idx in enumerate(topk_sorted, start=1):
            item_internal = int(i_vec[idx])
            item_ext = id2token_item[item_internal]
            rows.append((u_ext, item_ext, float(scores[idx]), rank))

recos_df = pd.DataFrame(rows, columns=["user_id", "item_id", "score", "rank"])

print("train interactions:", len(inter_train))
print("test (cold) interactions:", len(inter_test))
print("cold users:", len(cold_users))
print("recos rows:", len(recos_df))
recos_df.head(20)

In [ ]:
!pip install recbole>=1.2
!python -m recbole.quick_start.run_recbole --model=BPR --dataset=ml-1m

In [ ]:
!pip -q install kmeans-pytorch

In [ ]:
from recbole.quick_start import run_recbole

run_recbole(
    model='LightGCN',          # любая из 90+ моделей
    dataset='ml-1m',           # или путь к своему набору
    config_dict={              # приоритет над YAML и CLI
        'epochs': 50,
        'topk': 10,
        'neg_sampling': {'uniform': 1},
        'seed': 42,            # чтобы метрики не «плавали»
    }
)